## Example 1: Basic railroad diagram

In [ ]:
from railroad import Diagram, Terminal, NonTerminal, Sequence
from io import StringIO

# Example 1: Basic railroad diagram
print("Testing railroad-diagrams basic usage:")

# Create a simple SQL SELECT diagram
diagram = Diagram(
    Sequence(
        Terminal("SELECT"),
        NonTerminal("column_list"),
        Terminal("FROM"),
        NonTerminal("table_name")
    )
)

print(f"✓ Diagram created: {type(diagram).__name__}")

# Generate SVG
output = StringIO()
diagram.writeSvg(output.write)
svg = output.getvalue()

print(f"✓ SVG: {len(svg)} chars, Valid: {'<svg' in svg}")
import re
# Each Terminal and NonTerminal becomes a <text> label, in diagram order
labels = re.findall(r'<text[^>]*>([^<]*)</text>', svg)
print(f"✓ Labels: {labels}")
assert svg.startswith('<svg class="railroad-diagram"') and svg.rstrip().endswith('</svg>')
assert labels == ['SELECT', 'column_list', 'FROM', 'table_name']
assert svg.count('class="terminal ') == 2 and svg.count('class="non-terminal ') == 2

## Example 2: Choice and Optional elements

In [ ]:
from railroad import Diagram, Terminal, NonTerminal, Choice, Optional, Sequence
from io import StringIO

# Example 2: Choice and Optional elements
print("Testing railroad-diagrams Choice and Optional:")

# CREATE statement with choices
diagram = Diagram(
    Sequence(
        Terminal("CREATE"),
        Optional(Terminal("OR REPLACE")),
        Choice(0, Terminal("TABLE"), Terminal("VIEW"), Terminal("INDEX")),
        NonTerminal("identifier")
    )
)

print(f"✓ Complex diagram created")

# Generate SVG
output = StringIO()
diagram.writeSvg(output.write)
svg = output.getvalue()

print(f"✓ SVG length: {len(svg)} characters")
print(f"✓ Contains path: {'<path' in svg}")
import re
labels = re.findall(r'<text[^>]*>([^<]*)</text>', svg)
print(f"✓ Labels: {labels}")
assert labels == ['CREATE', 'OR REPLACE', 'TABLE', 'VIEW', 'INDEX', 'identifier']
assert '<path' in svg and svg.rstrip().endswith('</svg>')

## Example 3: ZeroOrMore for repetition

In [ ]:
from railroad import Diagram, Terminal, NonTerminal, Choice, ZeroOrMore, Sequence
from io import StringIO

# Example 3: ZeroOrMore for repetition
print("Testing railroad-diagrams ZeroOrMore:")

# Expression with operators
diagram = Diagram(
    Choice(0, NonTerminal("number"), NonTerminal("variable")),
    ZeroOrMore(
        Sequence(
            Choice(0, Terminal("+"), Terminal("-")),
            Choice(0, NonTerminal("number"), NonTerminal("variable"))
        )
    )
)

print(f"✓ Expression diagram created")

output = StringIO()
diagram.writeSvg(output.write)
svg = output.getvalue()
print(f"✓ Generated {len(svg)} characters")

import re
labels = re.findall(r'<text[^>]*>([^<]*)</text>', svg)
assert labels == ['number', 'variable', '+', '-', 'number', 'variable'], labels

## Example 4: OneOrMore for required repetition

In [ ]:
from railroad import Diagram, Terminal, NonTerminal, OneOrMore, Optional, Sequence
from io import StringIO

# Example 4: OneOrMore for required repetition
print("Testing railroad-diagrams OneOrMore:")

# Function definition
diagram = Diagram(
    Terminal("def"),
    NonTerminal("name"),
    Terminal("("),
    Optional(NonTerminal("params")),
    Terminal(")"),
    OneOrMore(NonTerminal("statement"))
)

print(f"✓ Function diagram created")

output = StringIO()
diagram.writeSvg(output.write)
svg = output.getvalue()

print(f"✓ Valid: {'<svg' in svg and '</svg>' in svg}")
import re
labels = re.findall(r'<text[^>]*>([^<]*)</text>', svg)
print(f"✓ Labels: {labels}")
assert labels == ['def', 'name', '(', 'params', ')', 'statement']

## Example 5: File output

In [ ]:
from railroad import Diagram, Terminal, NonTerminal, Choice, Sequence
import tempfile
import os

# Example 5: File output
print("Testing railroad-diagrams file output:")

# Mini-language grammar
diagram = Diagram(
    Choice(0,
        Sequence(Terminal("var"), NonTerminal("id"), Terminal("="), NonTerminal("expr")),
        Sequence(NonTerminal("id"), Terminal("("), Terminal(")")),
        Sequence(Terminal("if"), NonTerminal("cond"), Terminal("{"), Terminal("}"))
    )
)

print(f"✓ Grammar diagram created")

# Write to file
with tempfile.NamedTemporaryFile(mode='w', suffix='.svg', delete=False) as f:
    diagram.writeSvg(f.write)
    temp_file = f.name

print(f"✓ File: {os.path.getsize(temp_file)} bytes")
with open(temp_file) as f:
    content = f.read()
os.unlink(temp_file)

import re
labels = re.findall(r'<text[^>]*>([^<]*)</text>', content)
assert content.startswith('<svg') and content.rstrip().endswith('</svg>')
assert labels == ['var', 'id', '=', 'expr', 'id', '(', ')', 'if', 'cond', '{', '}'], labels